In [ ]:
import os
import torch

# Make sure we're in the right directory
os.chdir("/workspace/super-gan")

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify the dataset is ready
train_dir = "data/plantvillage/color_images/train"
assert os.path.isdir(train_dir), (
    f"Dataset not found at {train_dir}. "
    "Run 'bash setup.sh' in the terminal first."
)
num_classes = len(os.listdir(train_dir))
print(f"Dataset OK: {num_classes} classes found")

# Load Data

In [ ]:

from training_utils import (
    load_class_names,
    load_plant_balanced_image_file_sample,
    build_plant_dict,
    data_summary,
    summarize_image_attributes,
)

train_dir = "data/plantvillage/color_images/train"

class_names = load_class_names(train_dir)
plant_dict = build_plant_dict(class_names)
sample_image_files, sample_labels, sample_classes = load_plant_balanced_image_file_sample(
    train_dir, plant_dict, 500
)

# Create Dataset + Data Loader

In [ ]:
import dataset_utils
from torch.utils.data import DataLoader

dataset = dataset_utils.SuperResolutionDatasetWithLabels(image_files=sample_image_files, 
                                                         image_labels=sample_labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

print(f"Dataset: {len(dataset)} images")
print(f"Batches per epoch: {len(dataloader)}")

# Import Models

In [ ]:

#from training_loop import enhanced_training_loop
import importlib
from pathlib import Path
import os
import training_loop
import training_classes
import agricultural_sr
import generators
import discriminators
importlib.reload(training_loop)
importlib.reload(training_classes)
importlib.reload(agricultural_sr)
importlib.reload(discriminators)
importlib.reload(generators)

# Load Model

Note: **you need to update this** to reflect the proper file names and locations

Make sure to give the right name and checkpoint directory.


In [ ]:

def load_latest_generator_checkpoint(generator, 
                         checkpoint_dir="/checkpoints", 
                         model_name="enhanced_model"):
    """Load the latest checkpoint if available"""
    
    # expects /checkpoints/enhanced_model/latest_checkpoint.pt
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir = checkpoint_dir / model_name
    latest_path = checkpoint_dir / "latest_checkpoint.pt"
    
    if not latest_path.exists():
        print("No checkpoint found at latest path.")
        return 0 # failed
    
    checkpoint = torch.load(latest_path)
    
    generator.load_state_dict(checkpoint['generator_state_dict'])
    
    print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
    
    return 1 # success 

In [ ]:
# input parameters
model_name = "ag_gen_and_dis_model"
num_blocks = 4
total_epochs = 10
checkpoint_dir = "/checkpoints"

model_name = f"{model_name}_{num_blocks}blocks_total{total_epochs}epochs"

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# os.path.join(checkpoint_dir, model_name) ---> /content/drive/MyDrive/Crop Disease Detection Super Resolution/model_checkpoints/test_model_8blocks_total50epochs/
if not os.path.exists(os.path.join(checkpoint_dir, model_name)):
    print(f"Warning: No checkpoint directory found for {model_name}")

# replace with model of choice
generator = agricultural_sr.AgriculturalAttentionGenerator(num_residual_blocks=num_blocks).to(device)

# reload from latest checkpoint if available
status = load_latest_generator_checkpoint(generator,
                                    checkpoint_dir=checkpoint_dir,
                                    model_name=model_name)

if status == 0:
    print("No checkpoint loaded, please use a trained model.")

# Imports

In [ ]:
import importlib
importlib.reload(dataset_utils)

import importlib
importlib.reload(training_loop)

from torch.utils.data import DataLoader

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.cuda

# Define CNN Classifier 

In [ ]:
class SimpleCNNClassifier(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = F.adaptive_avg_pool2d(x, (4, 4))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Run Training

**Make sure you update the PATH and DIR and MODEL_NAME values.**

In [ ]:
#
# EXAMPLE PARAMETERS
#

NUM_CLASSES = 38
DEVICE = torch.device("cuda" if torch.cuda.is_available() and torch.cuda.is_available() else "cpu")
LEARNING_RATE = 1e-3
# NUM_TRAINING_EPOCHS = 10
# CHECKPOINT_PATH = "/content/drive/MyDrive/Crop Disease Detection Super Resolution/simple_classifier_hr_baseline.pth"
# CHECKPOINT_DIR = "/content/drive/MyDrive/Crop Disease Detection Super Resolution/model_checkpoints"
# MODEL_NAME = "baseline_classifer_low_res"


# Low Res Image Baseline

In [ ]:
import cnn_training

classifier = SimpleCNNClassifier(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

NUM_TRAINING_EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() and torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "/checkpoints"
MODEL_NAME = "baseline_classifer_low_res"

cnn_training.train_classifier(classifier,
                              optimizer,
                              criterion,
                              data_loader=DataLoader(dataset, batch_size=10, shuffle=True),
                              num_epochs=NUM_TRAINING_EPOCHS,
                              device=DEVICE,
                              checkpoint_path=CHECKPOINT_DIR,
                              model_name=MODEL_NAME,
                              low_res_input=True)


# High Res Image Baseline

In [ ]:
classifier = SimpleCNNClassifier(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

NUM_TRAINING_EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() and torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "/checkpoints"
MODEL_NAME = "baseline_classifer_high_res"

cnn_training.train_classifier(classifier,
                              optimizer,
                              criterion,
                              data_loader=DataLoader(dataset, batch_size=10, shuffle=True),
                              num_epochs=NUM_TRAINING_EPOCHS,
                              device=DEVICE,
                              checkpoint_path=CHECKPOINT_DIR,
                              model_name=MODEL_NAME,
                              low_res_input=False)

# GAN Model Based 

In [ ]:
classifier = SimpleCNNClassifier(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

NUM_TRAINING_EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() and torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "/checkpoints"
MODEL_NAME = "baseline_classifer_ag_gen"

cnn_training.train_classifier_w_generator(classifier,
                              generator,
                              optimizer,
                              criterion,
                              data_loader=DataLoader(dataset, batch_size=10, shuffle=True),
                              num_epochs=NUM_TRAINING_EPOCHS,
                              device=DEVICE,
                              checkpoint_path=CHECKPOINT_DIR,
                              model_name=MODEL_NAME)

# Next steps

1. Make sure to re-run the above GAN cell with each GAN model type (e.g. ag generator with 4, 8, 12 block, enhanced CNN, etc.)